# MVPDR+ Training Pipeline

Complete training pipeline for MVPDR+ (Multi-View Prototype-based Disease Recognition).

**Runtime:** GPU (T4 or better) — Runtime → Change runtime type → T4 GPU

**Stages:**
1. Setup & install dependencies
2. Download datasets (PlantDoc, PlantVillage)
3. Zero-shot CLIP baselines
4. MVPDR baseline training (1/5/10/20-shot)
5. MVPDR+ full model training
6. Ablation study
7. Aggregate results & generate figures

## 1. Setup

**Choose ONE method to get the code into Colab:**
- **Method A (recommended):** Upload a zip of the repo
- **Method B:** Clone from GitHub (requires public repo or auth)
- **Method C:** Mount Google Drive

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# === Method A: Upload zip (recommended for private repos) ===
# 1. On your local machine, run: cd ~/Downloads && zip -r mvpdr.zip mvpdr/ -x "mvpdr/.git/*" "mvpdr/data/*" "mvpdr/results/*" "mvpdr/__pycache__/*"
# 2. Then upload it here:
import os
from google.colab import files

if not os.path.exists('/content/mvpdr/pyproject.toml'):
    print("Upload mvpdr.zip:")
    uploaded = files.upload()
    !unzip -qo mvpdr.zip -d /content/
    # Handle nested directory (zip might create mvpdr/mvpdr/)
    if os.path.exists('/content/mvpdr/pyproject.toml'):
        print("Extracted successfully!")
    else:
        # Try one level deeper
        import glob
        toml = glob.glob('/content/*/pyproject.toml')
        if toml:
            parent = os.path.dirname(toml[0])
            print(f"Found project at {parent}")
            !mv {parent} /content/mvpdr
else:
    print("Project already exists at /content/mvpdr")

%cd /content/mvpdr

In [ ]:
# Install dependencies
!pip install -e . -q
!pip install kaggle -q
print("\nVerify install:")
!python -c "from mvpdr import clip; print('mvpdr package OK')"

In [ ]:
# Install dependencies
!pip install -e . -q
!pip install kaggle -q

## 2. Download Datasets

PlantDoc downloads automatically from GitHub.  
For PlantVillage, upload your `kaggle.json` or download manually.

In [ ]:
# Download PlantDoc (auto, from GitHub)
!python scripts/download_datasets.py --dataset plantdoc --output data/

In [ ]:
# Option A: Upload kaggle.json for PlantVillage auto-download
# Uncomment and run:
# from google.colab import files
# uploaded = files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !python scripts/download_datasets.py --dataset plantvillage --output data/

# Option B: Manual download from Kaggle
# 1. Download from https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
# 2. Upload the zip, then:
# !unzip plantvillage-dataset.zip -d /tmp/pv
# !mkdir -p data/plantvillage/images
# !cp -r /tmp/pv/*/color/* data/plantvillage/images/

print("Current datasets:")
!ls data/ 2>/dev/null || echo "No data/ directory yet"

In [ ]:
# Verify datasets
import os
for ds in ['plantdoc', 'plantvillage']:
    img_dir = f'data/{ds}/images'
    if os.path.exists(img_dir):
        classes = sorted(os.listdir(img_dir))
        total = sum(len(os.listdir(os.path.join(img_dir, c))) for c in classes if os.path.isdir(os.path.join(img_dir, c)))
        print(f'{ds}: {len(classes)} classes, {total} images')
    else:
        print(f'{ds}: NOT FOUND')

## 3. Zero-Shot CLIP Baselines

In [ ]:
!python scripts/evaluate_zeroshot.py \
    --root_path data/ \
    --output results/zeroshot/ \
    --dataset plantdoc

In [ ]:
# If PlantVillage is available:
# !python scripts/evaluate_zeroshot.py \
#     --root_path data/ \
#     --output results/zeroshot/ \
#     --dataset plantvillage

## 4. MVPDR Baseline Training

In [ ]:
# MVPDR baseline on PlantDoc — all shot settings
for shots in [1, 5, 10, 20]:
    print(f"\n{'='*60}")
    print(f"MVPDR Baseline | PlantDoc | {shots}-shot")
    print(f"{'='*60}")
    !python scripts/run_experiments.py \
        --root_path data/ \
        --stage mvpdr \
        --dataset plantdoc \
        --shots {shots} \
        --seeds 1

## 5. MVPDR+ Full Model Training

In [ ]:
# MVPDR+ on PlantDoc — all shot settings
for shots in [1, 5, 10, 20]:
    print(f"\n{'='*60}")
    print(f"MVPDR+ | PlantDoc | {shots}-shot")
    print(f"{'='*60}")
    !python scripts/run_experiments.py \
        --root_path data/ \
        --stage mvpdr_plus \
        --dataset plantdoc \
        --shots {shots} \
        --seeds 1

In [ ]:
# MVPDR+ on PlantVillage (if available)
# for shots in [1, 5, 10, 20]:
#     !python scripts/run_experiments.py \
#         --root_path data/ \
#         --stage mvpdr_plus \
#         --dataset plantvillage \
#         --shots {shots} \
#         --seeds 1

## 6. Ablation Study

In [ ]:
# Component ablation on PlantDoc 20-shot
!python scripts/run_experiments.py \
    --root_path data/ \
    --stage ablation \
    --seeds 1

## 7. Backbone Comparison

In [ ]:
# Compare RN50, RN101, ViT-B/32, ViT-B/16 on PlantDoc 20-shot
!python scripts/run_experiments.py \
    --root_path data/ \
    --stage backbone \
    --seeds 1

## 8. Aggregate Results

In [ ]:
!python scripts/aggregate_results.py \
    --results_dir results/ \
    --output results/summary/

In [ ]:
# Display results
with open('results/summary/results_table.md') as f:
    print(f.read())

In [ ]:
# Show figures
from IPython.display import Image, display
import os

for fig in ['accuracy_comparison.png', 'fewshot_curve.png']:
    path = f'results/summary/{fig}'
    if os.path.exists(path):
        print(f'\n{fig}:')
        display(Image(filename=path, width=700))

## 9. Download Results

In [ ]:
# Pack results for download
!tar czf mvpdr_results.tar.gz results/

from google.colab import files
files.download('mvpdr_results.tar.gz')

## 10. Multi-Seed Run (Optional — for confidence intervals)

Run the full suite with 3 seeds for standard deviation estimates.

In [ ]:
# Full experiment suite with 3 seeds (takes ~2-4 hours on T4)
# !python scripts/run_experiments.py \
#     --root_path data/ \
#     --stage all \
#     --seeds 1 2 3
# !python scripts/aggregate_results.py --results_dir results/